# Title

In [ ]:
import pandas as pd

product1_df = pd.read_csv("product.csv")
print(f"Dataset cargado: {product1_df.shape[0]:,} filas x {product1_df.shape[1]} columnas")
product1_df.head()

In [ ]:
product1_df.shape

In [ ]:
product_df = product1_df.head(200000).copy()

In [ ]:
product_df.head()

**1. Exploración y preparación de datos**

**1.1 Análisis Exploratorio (EDA)**

In [ ]:
product_df.dtypes

In [ ]:
missing_values = product_df.isnull().sum()
missing_percentage = 100 * product_df.isnull().sum() / len(product_df)

missing_df = pd.DataFrame({
    'Missing Value Count': missing_values,
    'Missing Value Percentage': missing_percentage
})

missing_df = missing_df[missing_df['Missing Value Count'] > 0].sort_values(by='Missing Value Percentage', ascending=False)

if not missing_df.empty:
    print('Valores faltantes por columna:')
    display(missing_df)
else:
    print('No hay valores faltantes en el DataFrame.')

In [ ]:
display(product_df.describe(include='all'))

In [ ]:
num_unique_users = product_df['user_id'].nunique()
print(f"Número de usuarios únicos: {num_unique_users}")

In [ ]:
event_type_distribution = product_df['title'].value_counts()
print("Distribución de tipos de eventos:")
display(event_type_distribution)

In [ ]:
site_version_distribution = product_df['site_version'].value_counts()
print("Distribución por versión de sitio:")
display(site_version_distribution)

In [ ]:
most_popular_products = product_df['product'].value_counts().head(3)
print("Los 3 productos más populares:")
display(most_popular_products)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.barplot(x=event_type_distribution.index, y=event_type_distribution.values, palette='viridis')
plt.title('Distribución de Tipos de Eventos')
plt.xlabel('Tipo de Evento')
plt.ylabel('Conteo')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
plt.pie(site_version_distribution.values, labels=site_version_distribution.index, autopct='%1.1f%%', startangle=90, colors=sns.color_palette('pastel'))
plt.title('Proporción de Versión de Sitio (Mobile vs. Desktop)')
plt.ylabel('') # Eliminar la etiqueta 'y' por defecto en pie charts
plt.show()

In [ ]:
plt.figure(figsize=(12, 7))
sns.barplot(x=most_popular_products.index, y=most_popular_products.values, palette='magma')
plt.title('Los 3 Productos Más Frecuentes')
plt.xlabel('Producto')
plt.ylabel('Conteo')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**1.2 Ingeniería de características**

In [ ]:
total_interactions_per_user = product_df.groupby('user_id').size().reset_index(name='total_interactions')

print("Primeras 5 filas del DataFrame de total_interactions por usuario:")
display(total_interactions_per_user.head())

In [ ]:
total_clicks_per_user = product_df[product_df['title'] == 'banner_click'].groupby('user_id').size().reset_index(name='total_clicks')

print("Primeras 5 filas del DataFrame de total_clicks por usuario:")
display(total_clicks_per_user.head())

In [ ]:
total_purchases_per_user = product_df[product_df['title'] == 'order'].groupby('user_id').size().reset_index(name='total_purchases')

print("Primeras 5 filas del DataFrame de total_purchases por usuario:")
display(total_purchases_per_user.head())

In [ ]:
merged_df = pd.merge(total_clicks_per_user, total_purchases_per_user, on='user_id', how='outer').fillna(0)

merged_df['click_to_purchase_ratio'] = merged_df.apply(
    lambda row: row['total_clicks'] / row['total_purchases'] if row['total_purchases'] > 0 else 0,
    axis=1
)

print("Primeras 5 filas del DataFrame con click_to_purchase_ratio:")
display(merged_df.head())

In [ ]:
mobile_events_per_user = product_df[product_df['site_version'] == 'mobile'].groupby('user_id').size().reset_index(name='mobile_events')
total_events_per_user = product_df.groupby('user_id').size().reset_index(name='total_events')

mobile_percentage_per_user = pd.merge(total_events_per_user, mobile_events_per_user, on='user_id', how='left').fillna(0)
mobile_percentage_per_user['mobile_percentage'] = (mobile_percentage_per_user['mobile_events'] / mobile_percentage_per_user['total_events']) * 100

print("Primeras 5 filas del DataFrame con el porcentaje de eventos móviles por usuario:")
display(mobile_percentage_per_user.head())

In [ ]:
desktop_events_per_user = product_df[product_df['site_version'] == 'desktop'].groupby('user_id').size().reset_index(name='desktop_events')

desktop_percentage_per_user = pd.merge(total_events_per_user, desktop_events_per_user, on='user_id', how='left').fillna(0)
desktop_percentage_per_user['desktop_percentage'] = (desktop_percentage_per_user['desktop_events'] / desktop_percentage_per_user['total_events']) * 100

print("Primeras 5 filas del DataFrame con el porcentaje de eventos de escritorio por usuario:")
display(desktop_percentage_per_user.head())

In [ ]:
product_df['time'] = pd.to_datetime(product_df['time'])
product_df_sorted = product_df.sort_values(by=['user_id', 'time'])

time_diffs = product_df_sorted.groupby('user_id')['time'].diff()
avg_time_between_events_per_user = time_diffs.groupby(product_df_sorted['user_id']).mean().dt.total_seconds() / 3600 # in hours

avg_time_between_events_per_user = avg_time_between_events_per_user.reset_index(name='avg_time_between_events_hours')

# Eliminar filas con valores NaN
avg_time_between_events_per_user = avg_time_between_events_per_user.dropna()

print("Primeras 5 filas del DataFrame con el tiempo promedio entre eventos por usuario (en horas) después de eliminar NaNs:")
display(avg_time_between_events_per_user.head())

In [ ]:
unique_products_viewed_per_user = product_df[product_df['title'] == 'banner_click'].groupby('user_id')['product'].nunique().reset_index(name='unique_products_viewed')

print("Primeras 5 filas del DataFrame con el número de productos únicos vistos (clics) por usuario:")
display(unique_products_viewed_per_user.head())

In [ ]:
unique_products_purchased_per_user = product_df[product_df['title'] == 'order'].groupby('user_id')['product'].nunique().reset_index(name='unique_products_purchased')

print("Primeras 5 filas del DataFrame con el número de productos únicos comprados por usuario:")
display(unique_products_purchased_per_user.head())

In [ ]:
import numpy as np

def shannon_entropy(product_counts):
    total_interactions = product_counts.sum()
    if total_interactions == 0:
        return 0
    proportions = product_counts / total_interactions
    # Filter out zero proportions to avoid log(0)
    proportions = proportions[proportions > 0]
    entropy = -np.sum(proportions * np.log(proportions))
    return entropy

# Calculate product counts per user
product_counts_per_user = product_df.groupby(['user_id', 'product']).size().unstack(fill_value=0)

# Apply Shannon entropy calculation for each user
avg_product_diversity = product_counts_per_user.apply(shannon_entropy, axis=1).reset_index(name='avg_product_diversity')

print("Primeras 5 filas del DataFrame con la diversidad de productos (índice de Shannon) por usuario:")
display(avg_product_diversity.head())

In [ ]:
total_shows_per_user = product_df[product_df['title'] == 'banner_show'].groupby('user_id').size().reset_index(name='total_shows')

# merged_df already contains total_clicks_per_user from a previous step.
# Merging total_shows_per_user with total_clicks_per_user (which is already available as total_clicks_per_user from previous steps)
show_click_ratio_df = pd.merge(total_shows_per_user, total_clicks_per_user, on='user_id', how='left').fillna(0)

show_click_ratio_df['show_to_click_ratio'] = show_click_ratio_df.apply(
    lambda row: row['total_clicks'] / row['total_shows'] if row['total_shows'] > 0 else 0,
    axis=1
)

print("Primeras 5 filas del DataFrame con la relación shows/clicks por usuario:")
display(show_click_ratio_df.head())

In [ ]:
product_df['date'] = product_df['time'].dt.date
days_active_per_user = product_df.groupby('user_id')['date'].nunique().reset_index(name='days_active')

print("Primeras 5 filas del DataFrame con el número de días activos por usuario:")
display(days_active_per_user.head())

In [ ]:
conversion_rate_df = pd.merge(total_purchases_per_user, total_interactions_per_user, on='user_id', how='left').fillna(0)
conversion_rate_df['conversion_rate'] = conversion_rate_df.apply(
    lambda row: row['total_purchases'] / row['total_interactions'] if row['total_interactions'] > 0 else 0,
    axis=1
)

print("Primeras 5 filas del DataFrame con la tasa de conversión por usuario:")
display(conversion_rate_df.head())

**2. Determinación del número óptimo de clústeres**


Consolidación y Preparación de Datos para Clustering
Primero, combinaremos todas las características de usuario que hemos generado (total_interactions, total_clicks, total_purchases, click_to_purchase_ratio, mobile_percentage, desktop_percentage, avg_time_between_events_hours, unique_products_viewed, unique_products_purchased, show_to_click_ratio, days_active, conversion_rate) en un único DataFrame. Esto es esencial para que el algoritmo K-means pueda considerar todas las dimensiones de comportamiento del usuario.

Nos aseguraremos de manejar los valores faltantes que puedan haber surgido de las fusiones de DataFrames, rellenándolos con 0. Luego, escalaremos las características para asegurar que ninguna domine el proceso de agrupamiento debido a su rango de valores.

In [ ]:
from functools import reduce
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Lista de DataFrames de características de usuario a fusionar
dfs_to_merge = [
    total_interactions_per_user,
    total_clicks_per_user[['user_id', 'total_clicks']],
    total_purchases_per_user[['user_id', 'total_purchases']],
    merged_df[['user_id', 'click_to_purchase_ratio']],
    mobile_percentage_per_user[['user_id', 'mobile_percentage']],
    desktop_percentage_per_user[['user_id', 'desktop_percentage']],
    avg_time_between_events_per_user,
    unique_products_viewed_per_user,
    unique_products_purchased_per_user,
    show_click_ratio_df[['user_id', 'show_to_click_ratio']],
    days_active_per_user,
    conversion_rate_df[['user_id', 'conversion_rate']]
]

# Fusionar todos los DataFrames usando reduce y 'outer' join para mantener a todos los usuarios
user_features_df = reduce(lambda left, right: pd.merge(left, right, on='user_id', how='outer'), dfs_to_merge)

# Rellenar valores NaN resultantes de las fusiones con 0
# Esto asume que un NaN significa que el usuario no tuvo esa actividad o característica
user_features_df = user_features_df.fillna(0)

print("Primeras 5 filas del DataFrame consolidado de características de usuario:")
display(user_features_df.head())
print(f"Forma del DataFrame de características de usuario: {user_features_df.shape}")
print("\nInformación sobre el DataFrame de características de usuario:")
user_features_df.info()

In [ ]:
# Separar el user_id antes de escalar
user_ids = user_features_df['user_id']
X = user_features_df.drop('user_id', axis=1)

# Escalar las características
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convertir el array escalado de nuevo a un DataFrame para facilitar el análisis
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns, index=user_ids)

print("Primeras 5 filas del DataFrame de características escaladas:")
display(X_scaled_df.head())

In [ ]:
# Calcular la inercia para diferentes valores de k
inertia = []
k_range = range(2, 11) # Evaluar k desde 2 hasta 10

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10) # n_init para evitar warnings
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

# Graficar el método del codo
plt.figure(figsize=(10, 6))
plt.plot(k_range, inertia, marker='o')
plt.title('Método del Codo para el Número Óptimo de Clústeres')
plt.xlabel('Número de Clústeres (k)')
plt.ylabel('Inercia')
plt.xticks(k_range)
plt.grid(True)
plt.show()

Documentación del Valor de K Recomendado
"El "codo" de la gráfica se localiza claramente en:

𝑘=4 

En este valor, la curva experimenta su cambio de dirección más brusco antes de entrar en una fase de rendimientos decrecientes.

Valor de  𝑘  Recomendado
Recomendación principal:  𝑘=4 .

Es el punto de equilibrio óptimo entre la cohesión de los grupos y la simplicidad del modelo.

In [ ]:
from sklearn.metrics import silhouette_score

# Calcular el Silhouette Score para diferentes valores de k
silhouette_scores = []
k_range = range(2, 11) # Evaluar k desde 2 hasta 10

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10) # n_init para evitar warnings
    kmeans.fit(X_scaled)
    score = silhouette_score(X_scaled, kmeans.labels_)
    silhouette_scores.append(score)

# Graficar k vs Silhouette Score
plt.figure(figsize=(10, 6))
plt.plot(k_range, silhouette_scores, marker='o')
plt.title('Silhouette Score para Diferentes Números de Clústeres')
plt.xlabel('Número de Clústeres (k)')
plt.ylabel('Silhouette Score')
plt.xticks(k_range)
plt.grid(True)
plt.show()

# Identificar el k con el mayor Silhouette Score
optimal_k_silhouette = k_range[silhouette_scores.index(max(silhouette_scores))]
print(f"El número óptimo de clústeres según el Silhouette Score es: {optimal_k_silhouette}")

In [ ]:
from sklearn.metrics import davies_bouldin_score

# Calcular el Davies-Bouldin Index para diferentes valores de k
davies_bouldin_scores = []
k_range = range(2, 11) # Evaluar k desde 2 hasta 10

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    score = davies_bouldin_score(X_scaled, kmeans.labels_)
    davies_bouldin_scores.append(score)

# Graficar k vs Davies-Bouldin Index
plt.figure(figsize=(10, 6))
plt.plot(k_range, davies_bouldin_scores, marker='o')
plt.title('Davies-Bouldin Index para Diferentes Números de Clústeres')
plt.xlabel('Número de Clústeres (k)')
plt.ylabel('Davies-Bouldin Index')
plt.xticks(k_range)
plt.grid(True)
plt.show()

# Identificar el k con el menor Davies-Bouldin Index
optimal_k_davies_bouldin = k_range[davies_bouldin_scores.index(min(davies_bouldin_scores))]
print(f"El número óptimo de clústeres según el Davies-Bouldin Index es: {optimal_k_davies_bouldin}")

Método del Codo: La gráfica del codo sugiere un valor de k = 4.

Silhouette Score: El Silhouette Score más alto se obtuvo con k = 8.

Davies-Bouldin Index: El Davies-Bouldin Index más bajo se obtuvo con k = 7.

Entre los tres, el Silhouette Score y el Davies-Bouldin Index son generalmente más confiables que el Método del Codo, ya que proporcionan una medida cuantitativa de la calidad del agrupamiento, reduciendo la subjetividad.

Considerando que el Silhouette Score sugiere k=8 y el Davies-Bouldin Index sugiere k=7, es común ver ligeras variaciones entre ellos.

El Davies-Bouldin Index de 7 es el valor más bajo, indicando la mejor separación de clústeres.

El valor de k que se usara en los algoritmos de clustering es: k = 7.